<a href="https://colab.research.google.com/github/ibrahimbarghout/robust-ecg-domain-generalization/blob/main/notebooks/07_demographic_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# PTB-XL Research Project
## 7. Demographic Baseline

#This notebook analyzes the demographic characteristics of the PTB-XL dataset
#and evaluates demographic consistency across the patient-independent
#training, validation, and test splits.

In [2]:
# ============================================
# 1. Mount Google Drive
# ============================================

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ============================================
# 2. Imports
# ============================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

Libraries imported successfully.


In [4]:
# ============================================
# 3. Project paths
# ============================================

PROJECT_PATH = "/content/drive/MyDrive/PTB-XL Research Project"

DATA_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
)

RESULTS_PATH = os.path.join(
    PROJECT_PATH,
    "results"
)

print("Dataset path:")
print(DATA_PATH)

print("\nDataset exists:", os.path.isdir(DATA_PATH))

print("\nResults path:")
print(RESULTS_PATH)

print("Results directory exists:", os.path.isdir(RESULTS_PATH))

Dataset path:
/content/drive/MyDrive/PTB-XL Research Project/data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3

Dataset exists: True

Results path:
/content/drive/MyDrive/PTB-XL Research Project/results
Results directory exists: True


In [5]:
# ============================================
# 4. Load PTB-XL metadata
# ============================================

database_path = os.path.join(
    DATA_PATH,
    "ptbxl_database.csv"
)

df = pd.read_csv(
    database_path,
    index_col="ecg_id"
)

print("ECG records:", len(df))
print("Unique patients:", df["patient_id"].nunique())

print("\nDemographic columns:")
print([
    "age",
    "sex",
    "height",
    "weight"
])

ECG records: 21799
Unique patients: 18869

Demographic columns:
['age', 'sex', 'height', 'weight']


In [6]:
# ============================================
# 5. Load diagnostic targets
# ============================================

targets_path = os.path.join(
    RESULTS_PATH,
    "ptbxl_diagnostic_targets.csv"
)

targets = pd.read_csv(
    targets_path,
    index_col="ecg_id"
)

print("Targets loaded:", os.path.isfile(targets_path))
print("Target shape:", targets.shape)

print("\nTarget columns:")
print(targets.columns.tolist())

Targets loaded: True
Target shape: (21799, 7)

Target columns:
['NORM', 'MI', 'STTC', 'CD', 'HYP', 'ABNORMAL', 'REFERENCE_NORMAL']


In [7]:
# ============================================
# 6. Load patient-independent split
# ============================================

split_path = os.path.join(
    RESULTS_PATH,
    "ptbxl_patient_independent_split.csv"
)

split_df = pd.read_csv(
    split_path,
    index_col="ecg_id"
)

print("Split loaded:", os.path.isfile(split_path))
print("Split records:", len(split_df))

print("\nSplit distribution:")
print(split_df["split"].value_counts())

Split loaded: True
Split records: 21799

Split distribution:
split
train         17418
test           2198
validation     2183
Name: count, dtype: int64


In [8]:
# ============================================
# 7. Demographic Data Completeness
# ============================================

demographic_columns = [
    "age",
    "sex",
    "height",
    "weight"
]

print("Missing demographic values:\n")

for column in demographic_columns:
    missing = df[column].isna().sum()
    percentage = missing / len(df) * 100

    print(
        f"{column:8s}: "
        f"{missing:5d} missing "
        f"({percentage:5.1f}%)"
    )

Missing demographic values:

age     :     0 missing (  0.0%)
sex     :     0 missing (  0.0%)
height  : 14825 missing ( 68.0%)
weight  : 12378 missing ( 56.8%)


In [9]:
# ============================================
# 8. Age Distribution
# ============================================

print("Age summary:")
print(df["age"].describe())

print("\nUnique age values:")
print(df["age"].nunique())

print("\nMinimum age:", df["age"].min())
print("Maximum age:", df["age"].max())

Age summary:
count    21799.000000
mean        62.769301
std         32.308813
min          2.000000
25%         50.000000
50%         62.000000
75%         72.000000
max        300.000000
Name: age, dtype: float64

Unique age values:
89

Minimum age: 2.0
Maximum age: 300.0


In [10]:
print("Age value distribution:")
print(df["age"].value_counts().sort_index())

print("\nPotentially invalid/special ages:")
print(df[df["age"] > 100][["age", "sex", "patient_id"]].head(30))

print("\nNumber of records with age > 100:")
print((df["age"] > 100).sum())

Age value distribution:
age
2.0        1
3.0        2
4.0        2
5.0        2
6.0        1
        ... 
86.0     178
87.0     230
88.0     149
89.0      98
300.0    293
Name: count, Length: 89, dtype: int64

Potentially invalid/special ages:
          age  sex  patient_id
ecg_id                        
108     300.0    1     11810.0
255     300.0    1      4867.0
256     300.0    1       565.0
279     300.0    0       437.0
282     300.0    0       437.0
346     300.0    1       565.0
351     300.0    1      5304.0
503     300.0    0      7535.0
540     300.0    0      8144.0
569     300.0    1      2936.0
604     300.0    1      3725.0
637     300.0    0      2145.0
646     300.0    1       520.0
689     300.0    0      3805.0
712     300.0    1       565.0
727     300.0    1       520.0
730     300.0    1       565.0
867     300.0    1     20236.0
873     300.0    1      5335.0
923     300.0    0      7408.0
932     300.0    0      4887.0
1065    300.0    1       627.0
1068    300.

In [11]:
# Create a cleaned age variable without modifying the original dataset
df["age_clean"] = df["age"].replace(300, np.nan)

print("Original age values:", df["age"].notna().sum())
print("Valid age values:", df["age_clean"].notna().sum())
print("Unknown/special ages:", df["age"].eq(300).sum())

print("\nCleaned age summary:")
print(df["age_clean"].describe())

Original age values: 21799
Valid age values: 21506
Unknown/special ages: 293

Cleaned age summary:
count    21506.000000
mean        59.537245
std         16.758773
min          2.000000
25%         50.000000
50%         61.000000
75%         72.000000
max         89.000000
Name: age_clean, dtype: float64


In [12]:
print("Sex distribution:")
print(df["sex"].value_counts(dropna=False))

print("\nSex percentages:")
print(
    df["sex"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

Sex distribution:
sex
0    11354
1    10445
Name: count, dtype: int64

Sex percentages:
sex
0    52.08
1    47.92
Name: proportion, dtype: float64


In [13]:
print("Sex coding in PTB-XL:")
print("0 =", df["sex"].eq(0).sum())
print("1 =", df["sex"].eq(1).sum())

print("\nSex by diagnostic target:")

for target in ["NORM", "MI", "STTC", "CD", "HYP"]:
    print(f"\n{target}")
    print(
        pd.crosstab(
            df["sex"],
            targets[target],
            normalize="index"
        ).round(3)
    )

Sex coding in PTB-XL:
0 = 11354
1 = 10445

Sex by diagnostic target:

NORM
NORM      0      1
sex               
0     0.614  0.386
1     0.509  0.491

MI
MI       0      1
sex              
0    0.700  0.300
1    0.803  0.197

STTC
STTC      0      1
sex               
0     0.774  0.226
1     0.744  0.256

CD
CD       0      1
sex              
0    0.736  0.264
1    0.818  0.182

HYP
HYP      0      1
sex              
0    0.866  0.134
1    0.892  0.108


In [14]:
print("HEIGHT")
print("=" * 50)

print("Missing:", df["height"].isna().sum())
print("Available:", df["height"].notna().sum())

print("\nSummary:")
print(df["height"].describe())

print("\nWEIGHT")
print("=" * 50)

print("Missing:", df["weight"].isna().sum())
print("Available:", df["weight"].notna().sum())

print("\nSummary:")
print(df["weight"].describe())

HEIGHT
Missing: 14825
Available: 6974

Summary:
count    6974.000000
mean      166.702323
std        10.867321
min         6.000000
25%       160.000000
50%       166.000000
75%       174.000000
max       209.000000
Name: height, dtype: float64

WEIGHT
Missing: 12378
Available: 9421

Summary:
count    9421.000000
mean       70.995223
std        15.878803
min         5.000000
25%        60.000000
50%        70.000000
75%        80.000000
max       250.000000
Name: weight, dtype: float64


In [15]:
print("Potentially unusual height values:")
print(df.loc[df["height"].notna(), "height"].sort_values().head(20))
print("\nHighest height values:")
print(df.loc[df["height"].notna(), "height"].sort_values().tail(20))

print("\nPotentially unusual weight values:")
print(df.loc[df["weight"].notna(), "weight"].sort_values().head(20))
print("\nHighest weight values:")
print(df.loc[df["weight"].notna(), "weight"].sort_values().tail(20))

Potentially unusual height values:
ecg_id
9317       6.0
9384       6.0
15207      6.0
6266      66.0
6430      67.0
6481      80.0
10289     85.0
15351     90.0
15514     90.0
16198     90.0
18173     93.0
14263     95.0
13945     97.0
9736     100.0
2544     104.0
12390    109.0
2696     116.0
3150     116.0
4039     120.0
6598     120.0
Name: height, dtype: float64

Highest height values:
ecg_id
4529     192.0
12544    193.0
2206     193.0
2214     193.0
1383     193.0
3004     193.0
6886     193.0
9855     193.0
3091     193.0
6370     193.0
11987    193.0
14068    194.0
8047     195.0
12717    195.0
6529     195.0
16395    196.0
3281     196.0
430      197.0
12592    200.0
18135    209.0
Name: height, dtype: float64

Potentially unusual weight values:
ecg_id
7945      5.0
6059      5.0
10289    12.0
13666    15.0
9736     16.0
4695     17.0
13945    17.0
16198    19.0
12390    19.0
14263    20.0
6598     20.0
3961     25.0
4001     31.0
2205     31.0
1625     32.0
5740     32.0
17

In [16]:
# Inspect records with clearly suspicious anthropometric values

suspicious_demo = df[
    (df["height"].notna() & (df["height"] < 120)) |
    (df["weight"].notna() & (df["weight"] < 30))
][
    ["patient_id", "age", "sex", "height", "weight"]
].sort_values(["height", "weight"])

print("Number of suspicious anthropometric records:", len(suspicious_demo))

display(suspicious_demo.head(50))

Number of suspicious anthropometric records: 24


,patient_id,age,sex,height,weight
ecg_id,,,,,
9317,17006.0,64.0,0,6.0,NaN
9384,15676.0,79.0,1,6.0,NaN
15207,9105.0,86.0,1,6.0,NaN
6266,20132.0,60.0,1,66.0,NaN
6430,4070.0,46.0,1,67.0,NaN
6481,2133.0,76.0,0,80.0,NaN
10289,5336.0,2.0,0,85.0,12.0
16198,7465.0,6.0,0,90.0,19.0
15351,12631.0,84.0,1,90.0,NaN


In [17]:
print("Suspicious records with age:")

display(
    suspicious_demo[
        ["patient_id", "age", "sex", "height", "weight"]
    ].sort_values("age").head(50)
)

Suspicious records with age:


,patient_id,age,sex,height,weight
ecg_id,,,,,
10289,5336.0,2.0,0,85.0,12.0
9736,1681.0,3.0,1,100.0,16.0
14263,3751.0,4.0,1,95.0,20.0
13666,5548.0,4.0,1,140.0,15.0
12390,5878.0,5.0,0,109.0,19.0
13945,4587.0,5.0,0,97.0,17.0
16198,7465.0,6.0,0,90.0,19.0
6598,6967.0,8.0,1,120.0,20.0
3961,7050.0,9.0,1,137.0,25.0


In [18]:
# Create cleaned anthropometric variables
# Raw height and weight are preserved unchanged.

df["height_clean"] = df["height"]
df["weight_clean"] = df["weight"]

# Clearly impossible values
df.loc[df["height_clean"] <= 0, "height_clean"] = np.nan
df.loc[df["weight_clean"] <= 0, "weight_clean"] = np.nan

# Flag clearly implausible adult measurements
adult = df["age_clean"] >= 18

height_implausible = adult & (
    (df["height_clean"] < 120) |
    (df["height_clean"] > 220)
)

weight_implausible = adult & (
    (df["weight_clean"] < 30) |
    (df["weight_clean"] > 250)
)

df.loc[height_implausible, "height_clean"] = np.nan
df.loc[weight_implausible, "weight_clean"] = np.nan

print("Height values removed as implausible:", height_implausible.sum())
print("Weight values removed as implausible:", weight_implausible.sum())

print("\nCleaned height summary:")
print(df["height_clean"].describe())

print("\nCleaned weight summary:")
print(df["weight_clean"].describe())

Height values removed as implausible: 12
Weight values removed as implausible: 3

Cleaned height summary:
count    6962.000000
mean      166.869003
std         9.970237
min        85.000000
25%       160.000000
50%       166.000000
75%       174.000000
max       209.000000
Name: height_clean, dtype: float64

Cleaned weight summary:
count    9418.000000
mean       71.014971
std        15.842403
min        12.000000
25%        60.000000
50%        70.000000
75%        80.000000
max       250.000000
Name: weight_clean, dtype: float64


In [19]:
print("Missing demographic data by diagnostic target")
print("=" * 60)

for target in ["NORM", "MI", "STTC", "CD", "HYP"]:

    subset = targets[target] == 1

    print(f"\n{target} (n={subset.sum():,})")

    for variable in ["age_clean", "sex", "height_clean", "weight_clean"]:

        missing = df.loc[subset, variable].isna().sum()
        percentage = missing / subset.sum() * 100

        print(
            f"{variable:15s}: "
            f"{missing:5,d} missing "
            f"({percentage:5.1f}%)"
        )

Missing demographic data by diagnostic target

NORM (n=9,514)
age_clean      :    31 missing (  0.3%)
sex            :     0 missing (  0.0%)
height_clean   : 6,264 missing ( 65.8%)
weight_clean   : 4,251 missing ( 44.7%)

MI (n=5,469)
age_clean      :   121 missing (  2.2%)
sex            :     0 missing (  0.0%)
height_clean   : 4,065 missing ( 74.3%)
weight_clean   : 3,927 missing ( 71.8%)

STTC (n=5,235)
age_clean      :   136 missing (  2.6%)
sex            :     0 missing (  0.0%)
height_clean   : 3,350 missing ( 64.0%)
weight_clean   : 3,238 missing ( 61.9%)

CD (n=4,898)
age_clean      :   142 missing (  2.9%)
sex            :     0 missing (  0.0%)
height_clean   : 3,334 missing ( 68.1%)
weight_clean   : 3,081 missing ( 62.9%)

HYP (n=2,649)
age_clean      :    56 missing (  2.1%)
sex            :     0 missing (  0.0%)
height_clean   : 1,616 missing ( 61.0%)
weight_clean   : 1,576 missing ( 59.5%)


In [20]:
print("Missingness comparison: target-positive vs target-negative")
print("=" * 70)

for target in ["NORM", "MI", "STTC", "CD", "HYP"]:

    print(f"\n{'=' * 20} {target} {'=' * 20}")

    positive = targets[target] == 1
    negative = targets[target] == 0

    for variable in ["age_clean", "height_clean", "weight_clean"]:

        pos_missing = df.loc[positive, variable].isna().mean() * 100
        neg_missing = df.loc[negative, variable].isna().mean() * 100

        difference = pos_missing - neg_missing

        print(
            f"{variable:15s} | "
            f"Positive: {pos_missing:5.1f}% | "
            f"Negative: {neg_missing:5.1f}% | "
            f"Difference: {difference:+5.1f}%"
        )

Missingness comparison: target-positive vs target-negative

==================== NORM ====================
age_clean       | Positive:   0.3% | Negative:   2.1% | Difference:  -1.8%
height_clean    | Positive:  65.8% | Negative:  69.8% | Difference:  -3.9%
weight_clean    | Positive:  44.7% | Negative:  66.2% | Difference: -21.5%

==================== MI ====================
age_clean       | Positive:   2.2% | Negative:   1.1% | Difference:  +1.2%
height_clean    | Positive:  74.3% | Negative:  66.0% | Difference:  +8.4%
weight_clean    | Positive:  71.8% | Negative:  51.8% | Difference: +20.0%

==================== STTC ====================
age_clean       | Positive:   2.6% | Negative:   0.9% | Difference:  +1.7%
height_clean    | Positive:  64.0% | Negative:  69.3% | Difference:  -5.4%
weight_clean    | Positive:  61.9% | Negative:  55.2% | Difference:  +6.7%

==================== CD ====================
age_clean       | Positive:   2.9% | Negative:   0.9% | Difference:  +2.0%
hei

In [21]:
print("Age distribution by diagnostic target")
print("=" * 70)

for target in ["NORM", "MI", "STTC", "CD", "HYP"]:

    positive = targets[target] == 1

    age_values = df.loc[positive, "age_clean"].dropna()

    print(f"\n{target} (n={len(age_values):,})")
    print(
        f"Mean   : {age_values.mean():.1f}\n"
        f"Median : {age_values.median():.1f}\n"
        f"Std    : {age_values.std():.1f}\n"
        f"Min    : {age_values.min():.1f}\n"
        f"Max    : {age_values.max():.1f}"
    )

Age distribution by diagnostic target

NORM (n=9,483)
Mean   : 52.1
Median : 54.0
Std    : 17.2
Min    : 2.0
Max    : 89.0

MI (n=5,348)
Mean   : 66.3
Median : 67.0
Std    : 12.5
Min    : 15.0
Max    : 89.0

STTC (n=5,099)
Mean   : 66.3
Median : 67.0
Std    : 13.5
Min    : 10.0
Max    : 89.0

CD (n=4,756)
Mean   : 65.3
Median : 67.0
Std    : 15.0
Min    : 8.0
Max    : 89.0

HYP (n=2,593)
Mean   : 65.9
Median : 67.0
Std    : 13.9
Min    : 4.0
Max    : 89.0


In [23]:
# Save cleaned demographic variables

demographics_clean = df[
    [
        "patient_id",
        "age",
        "age_clean",
        "sex",
        "height",
        "height_clean",
        "weight",
        "weight_clean"
    ]
].copy()

# Put ECG ID back as a regular column
demographics_clean.insert(
    0,
    "ecg_id",
    demographics_clean.index
)

demographics_path = os.path.join(
    RESULTS_PATH,
    "ptbxl_cleaned_demographics.csv"
)

demographics_clean.to_csv(
    demographics_path,
    index=False
)

print("Saved cleaned demographics to:")
print(demographics_path)

print("File exists:", os.path.exists(demographics_path))
print("Shape:", demographics_clean.shape)

print("\nFirst 5 rows:")
display(demographics_clean.head())

Saved cleaned demographics to:
/content/drive/MyDrive/PTB-XL Research Project/results/ptbxl_cleaned_demographics.csv
File exists: True
Shape: (21799, 9)

First 5 rows:


,ecg_id,patient_id,age,age_clean,sex,height,height_clean,weight,weight_clean
ecg_id,,,,,,,,,
1,1,15709.0,56.0,56.0,1,NaN,NaN,63.0,63.0
2,2,13243.0,19.0,19.0,0,NaN,NaN,70.0,70.0
3,3,20372.0,37.0,37.0,1,NaN,NaN,69.0,69.0
4,4,17014.0,24.0,24.0,0,NaN,NaN,82.0,82.0
5,5,17448.0,19.0,19.0,1,NaN,NaN,70.0,70.0
